# Application 1 — Sentiment Analysis

* * *

So far we have worked through the building blocks of an NLP pipeline. In this notebook we put those pieces together for our first real *application*: **sentiment analysis** — classifying or scoring how positive or negative a piece of text is.

Sentiment analysis is one of the most widely-used NLP applications in social science and industry:
- product / restaurant review analysis
- monitoring public opinion on social media (e.g., reactions to a policy or product launch)
- customer-service ticket triage (angry customer → escalate)
- finance — news / earnings-call sentiment as a market signal
- political science — tone of legislative speech, campaign messaging


# Recap — the basic NLP toolkit

Before we move on to applications, let's take stock of the basic NLP tasks we've already covered. Each one is a transformation that takes raw text a step closer to something a model can work with.

<img src='./images/diagram.png' alt="cv" width="1200">


Sentiment analysis sits squarely on top of this stack. A simple sentiment model is just a classifier (or regressor) trained on text features, with positive/negative labels (or scores). Once you have a sentiment score per text, you can plug it into a lot of social-science workflows — for example:

In the rest of this notebook we walk through three increasingly hands-on layers of sentiment analysis:
- Part 1 — try off-the-shelf models (TextBlob and modern Transformer-based sentiment models for English and Korean) and see what you can get for free.
- Part 2 — build your own classifier from labeled review data. 
- Part 3 — fine-tune a pre-trained Transformer (DistilBERT) end-to-end and compare it against the simpler baselines from Part 2.

# Setup

In [ ]:
# !pip install textblob transformers datasets scikit-learn scipy
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context("talk")
import warnings
warnings.filterwarnings("ignore")

# Part 1 — Off-the-shelf sentiment models

Before building anything, you should know what you can get *for free*. For many social-science applications an off-the-shelf model is good enough — and even if you do train your own, it is the right baseline to compare against.

## 1.1  TextBlob — the classic lexicon-based scorer

TextBlob's sentiment is based on a hand-crafted lexicon (lists of words tagged with polarity / subjectivity scores; mainly adjectives). It returns two numbers per text:

- `polarity` ∈ [-1, +1] — how negative / positive
- `subjectivity` ∈ [0, 1] — how opinion-like vs. fact-like

It is:
- fast (no model loading, no GPU)
- interpretable (you can look up exactly which words contributed)
- English-only and **very brittle** — it does not understand negation well, has no context, no sarcasm. Treat the polarity score as a rough heuristic, not as ground truth.

In [ ]:
from textblob import TextBlob

samples = [
    "I absolutely loved this movie — best film of the year!",
    "Terrible service, cold food, will never come back.",
    "The package arrived on time.",
    "I do not like this product at all.",     # negation
    "Oh great, another delayed flight.",       # sarcasm
]

for s in samples:
    pol  = TextBlob(s).sentiment.polarity
    subj = TextBlob(s).sentiment.subjectivity
    print(f"  polarity={pol:+.2f}  subj={subj:.2f}  |  {s}")

Notice how TextBlob gets the obvious cases right but fails on negation ("I do not like…" → near-zero polarity) and on sarcasm ("Oh great…" → positive). This is why modern transformer-based sentiment models have largely replaced lexicon-based ones for serious work.

## 1.2  Modern transformer-based models (English)

A transformer fine-tuned for sentiment is dramatically better than TextBlob. It is trained end-to-end on labeled sentiment data, so it learns negation, intensifiers, idioms, and (some) sarcasm directly from examples. The cost is that it needs to download model weights and is slower.

A few solid, well-maintained options on the HuggingFace Hub:

| Model | Output | Notes |
|---|---|---|
| `distilbert-base-uncased-finetuned-sst-2-english` | binary (POS/NEG) + score | classic, small, fast |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` | 3-class (neg/neu/pos) | trained on tweets, very current |
| `siebert/sentiment-roberta-large-english` | binary (POS/NEG) | strong general-domain model |
| `nlptown/bert-base-multilingual-uncased-sentiment` | 1-5 stars | multilingual, includes Korean |

We'll use the HuggingFace `pipeline` interface — give it text, get predictions back.

In [ ]:
from transformers import pipeline

sentiment_en = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
)

for s in samples:
    out = sentiment_en(s)[0]
    print(f"  {out['label']:8s}  ({out['score']:.2f})  |  {s}")

## 1.3  Korean sentiment models

For Korean text, you need a model that has been trained on Korean. Practical options:

| Model | Output | Trained on |
|---|---|---|
| `nlptown/bert-base-multilingual-uncased-sentiment` | 1-5 stars | multilingual product reviews — works on Korean directly |
| `WhitePeak/bert-base-cased-Korean-sentiment` | binary (POS/NEG) | Korean shopping reviews |
| `snunlp/KR-FinBert-SC` | 3-class (neg/neu/pos) | Korean financial / news |
| `matthewburke/korean_sentiment` | binary | Korean reviews |


In [ ]:
ko_samples = [
    "이 영화 정말 재미있었어요! 강추합니다.",
    "기대했는데 너무 실망스러웠다. 시간 낭비.",
    "그냥 그랬어요. 볼만은 했네요.",
    "재미가 없지는 않았어요.",       # negation
    "정말 최고의 영화였어요. 잠을 아주 푹 잘 수 있었거든요.",  # sarcasm
]

# Multilingual 1-5 star model — runs on Korean out of the box
sentiment_ko = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

for s in ko_samples:
    out = sentiment_ko(s)[0]
    print(f"  {out['label']:8s}  ({out['score']:.2f})  |  {s}")

# Part 2 — Build your own sentiment model

Now we'll train a sentiment model from scratch on labeled data. The actual training loop is just our familiar classification recipe from the last tutorial:

```
text → vectorize (e.g., tf-idf) → fit classifier → evaluate
```

In the previous tutorial we held the labels fixed and varied **X** — different feature representations (bag-of-words, tf-idf, embeddings) — and watched how each choice changed performance. In this part we flip the experiment: we hold the *text and the features* fixed and vary **y** — the thing the model is asked to predict. The point is that the same raw signal can be turned into very different prediction tasks, and *how you shape `y`* often matters more than which classifier you pick.

## The data — EmoBank (sentences with continuous Valence scores)

For this experiment we need a dataset where the underlying sentiment signal is **continuous**, so we can carve it into different shapes of `y`. We'll use [EmoBank](https://github.com/JULIELab/EmoBank) — ~10k English sentences from news, fiction, blogs, etc., each annotated by multiple human raters on the **Valence** dimension (positive ↔ negative). Because the published score is the *average* across annotators, it lives on a continuous 1-5 scale — exactly what we need to build three different sentiment formulations from the *same* signal:

1. **binary** — positive vs. negative (threshold the Valence score)
2. **regression** — predict the continuous Valence score directly
3. **3-way** — low / medium / high Valence

Same `X`, three different `y`s, three different models. Watch how the metric, the model, and even the *kind of error* changes as we move between them.

In [ ]:
import os, urllib.request

# EmoBank — 10k English sentences annotated for continuous Valence / Arousal / Dominance.
# We download the CSV once and cache it locally.
EMOBANK_URL  = "https://raw.githubusercontent.com/JULIELab/EmoBank/master/corpus/emobank.csv"
EMOBANK_PATH = "../data/emobank.csv"

os.makedirs(os.path.dirname(EMOBANK_PATH), exist_ok=True)
if not os.path.exists(EMOBANK_PATH):
    urllib.request.urlretrieve(EMOBANK_URL, EMOBANK_PATH)

emo = pd.read_csv(EMOBANK_PATH)
# Keep just what we need: the sentence and the Valence (V) score (continuous, 1-5).
df = emo[["text", "V"]].rename(columns={"V": "valence"}).dropna().reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
print(df["valence"].describe().round(3))

plt.figure(figsize=(7, 4))
plt.hist(df["valence"], bins=40, edgecolor="white")
plt.axvline(3.0, color="k", linestyle="--", lw=1, label="neutral (V=3)")
plt.xlabel("Valence  (1 = very negative,  5 = very positive)")
plt.ylabel("# sentences")
plt.title("EmoBank — distribution of Valence scores")
plt.legend(); plt.tight_layout(); plt.show()

The distribution is sharply peaked near 3 (neutral) — most sentences in real corpora carry only mild affect. Let's peek at a few sentences at different ends of the Valence scale to sanity-check the labels.

In [ ]:
bands = [
    ("very negative  (V ≤ 2.0)",             df["valence"] <= 2.0),
    ("negative       (2.0 < V ≤ 2.6)",       (df["valence"] > 2.0) & (df["valence"] <= 2.6)),
    ("neutral        (2.6 < V < 3.4)",       (df["valence"] > 2.6) & (df["valence"] < 3.4)),
    ("positive       (3.4 ≤ V < 4.0)",       (df["valence"] >= 3.4) & (df["valence"] < 4.0)),
    ("very positive  (V ≥ 4.0)",             df["valence"] >= 4.0),
]
for label, mask in bands:
    sub = df.loc[mask]
    if len(sub):
        row = sub.sample(1, random_state=0).iloc[0]
        print(f"\n{label}  (n={len(sub)})  V={row['valence']:.2f}\n  {row['text'][:200]}")

## Build train/test split
<img src='./images/datasplit.png' alt="cv" width="800">

image from [here](https://www.lightly.ai/blog/train-test-validation-split)

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42,
)
print(f"Train: {len(train_idx)}    Test: {len(test_idx)}")

And we'll use the same tf-idf vectorizer for everything. **Fit on the training split only.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True, stop_words="english",
    min_df=2, max_df=0.95, ngram_range=(1, 2),
)

X_train = tfidf.fit_transform(df.loc[train_idx, "text"])
X_test  = tfidf.transform(df.loc[test_idx,  "text"])
print(f"Tf-idf shape:  train={X_train.shape}   test={X_test.shape}")

## Formulation 1 — Binary: positive vs. negative

The simplest framing. *"Is this sentence positive?"* Yes / no.

Decisions you have to make when constructing `y`:

- **Where is the cutoff?** EmoBank Valence is centered around 3 (neutral). A natural choice: `V > 3` → positive, `V < 3` → negative.
- **What do we do with the middle?** A *lot* of sentences sit very close to V = 3. Two options:
    1. *drop near-neutral rows* (e.g., `|V − 3| < 0.2`) — cleaner labels but throws away data
    2. *keep everything and split exactly at 3* — keeps the data but the model has to learn from many essentially-neutral examples that could go either way
- It's a **classification** problem → `y` is integers, model is a classifier.

We'll drop the near-neutral band for clean labels, and use `LogisticRegression` as the classifier.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

v_train = df.loc[train_idx, "valence"].values
v_test  = df.loc[test_idx,  "valence"].values
print(f"Before dropping neutral sentences:  train={len(v_train)}  test={len(v_test)}")

# --- construct y for the binary task ---
NEUTRAL_BAND = 0.2                              # drop rows with |V − 3| < 0.2
mask_train = np.abs(v_train - 3) >= NEUTRAL_BAND
mask_test  = np.abs(v_test  - 3) >= NEUTRAL_BAND
print(f"After dropping neutral sentences:  train={mask_train.sum()}  test={mask_test.sum()}")
y_train_bin = (v_train[mask_train] > 3).astype(int) # 1 (positive) if V > 3, else 0 (negative)
y_test_bin  = (v_test[mask_test]   > 3).astype(int)

# subset X the same way
X_train_bin = X_train[mask_train]
X_test_bin  = X_test[mask_test]

# print shapes
print(f"X_train_bin.shape = {X_train_bin.shape}   ← {X_train_bin.shape[0]} training docs × {X_train_bin.shape[1]} tf-idf features (sparse)")
print(f"y_train_bin.shape = {y_train_bin.shape}        ← one label (0=neg, 1=pos) per training doc")
print(f"X_test_bin.shape  = {X_test_bin.shape}    ← {X_test_bin.shape[0]} test docs × {X_test_bin.shape[1]} tf-idf features (same vocab as train)")
print(f"y_test_bin.shape  = {y_test_bin.shape}         ← one label per test doc")

In [ ]:
# visualize x and y data
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

N_ROWS = 200
N_COLS = 200

X_sub = X_train_bin[:N_ROWS, :N_COLS].toarray()
y_sub = y_train_bin[:N_ROWS].reshape(-1, 1)

fig, (ax_x, ax_y, ax_lg) = plt.subplots(
    1, 3, figsize=(13, 8),
    gridspec_kw={"width_ratios": [20, 1, 0.6], "wspace": 0.4},
)

# --- X_train_bin (left): sparsity pattern ---
ax_x.imshow(X_sub > 0, aspect="auto", cmap="Greys", interpolation="nearest")
ax_x.set_title(f"X_train_bin  (first {N_ROWS} rows × {N_COLS} cols, black = non-zero tf-idf)")
ax_x.set_xlabel("vocab column")
ax_x.set_ylabel("row (document)")

# --- y_train_bin (middle): label column ---
cmap = ListedColormap(["#d9534f", "#5cb85c"])
ax_y.imshow(y_sub, aspect="auto", cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
ax_y.set_title("y_train_bin")
ax_y.set_xticks([])
ax_y.set_yticks([])

# --- legend (far right, own axis) ---
ax_lg.axis("off")
ax_lg.legend(
    handles=[Patch(color="#d9534f", label="0 = neg"),
             Patch(color="#5cb85c", label="1 = pos")],
    loc="center left", fontsize=10, frameon=False,
)

plt.show()


In [ ]:
# --- fit & evaluate ---
clf_bin = LogisticRegression(max_iter=2000).fit(X_train_bin, y_train_bin)
y_pred_bin = clf_bin.predict(X_test_bin)
print(f"\nAccuracy: {accuracy_score(y_test_bin, y_pred_bin):.3f}")
print(classification_report(y_test_bin, y_pred_bin, target_names=["negative", "positive"]))

## Formulation 2 — Regression: continuous score in [0, 1]

This is the most natural framing for EmoBank — the labels themselves are continuous, so we just predict the number directly. Useful when:

- downstream code wants a continuous "how positive" signal (e.g., averaging sentiment across many sentences in a document, or correlating with another variable),
- you care about the *ranking* (which sentences are most positive), not just the category.

Decisions for `y`:

- Map Valence (1-5) to a number. The standard choice is to **normalize linearly**: `(V - 1) / 4` puts it in `[0, 1]`. This is *not strictly necessary* — predicting `V` directly on its original 1-5 scale gives the same Pearson `r` (linear transforms don't change correlation) and an MAE that's exactly 4× larger. We still do it because 
    - (a) `[0, 1]` is more intuitive to read ("0.7 ≈ 70% positive"), 
    - (b) some downstream models (e.g., neural nets) expect targets in `[0, 1]`.
- It's a **regression** problem → `y` is floats, model is a regressor. We use `Ridge` (linear regression with L2) — fast and a sensible default for tf-idf features.

The metric is no longer accuracy. We'll use **MAE** (mean absolute error, lower is better) and **Pearson r** (correlation between predicted and true scores, higher is better).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

# --- construct y for regression: normalize Valence (1-5) → [0, 1] ---
y_train_reg = (v_train - 1) / 4.0
y_test_reg  = (v_test  - 1) / 4.0


print(f"  y_train_reg sample : {y_train_reg[:5].round(3)}")
print(f"  range              : [{y_train_reg.min():.3f}, {y_train_reg.max():.3f}]")
print(f"  mean ± std         : {y_train_reg.mean():.3f} ± {y_train_reg.std():.3f}")

In [ ]:
# visualize x and y data
import matplotlib.pyplot as plt
import numpy as np

N_ROWS = 200
N_COLS = 200

X_sub = X_train[:N_ROWS, :N_COLS].toarray()
y_sub = y_train_reg[:N_ROWS].reshape(-1, 1)

fig, (ax_x, ax_y, ax_cb) = plt.subplots(
    1, 3, figsize=(13, 8),
    gridspec_kw={"width_ratios": [20, 1, 0.6], "wspace": 0.4},
)

# --- X_train (left) ---
ax_x.imshow(X_sub > 0, aspect="auto", cmap="Greys", interpolation="nearest")
ax_x.set_title(f"X_train  (first {N_ROWS} rows × {N_COLS} cols, black = non-zero tf-idf)")
ax_x.set_xlabel("vocab column")
ax_x.set_ylabel("row (document)")

# --- y_train_reg (middle) ---
im = ax_y.imshow(y_sub, aspect="auto", cmap="RdYlGn",
                 vmin=0, vmax=1, interpolation="nearest")
ax_y.set_title("y_train_reg")
ax_y.set_xticks([])
ax_y.set_yticks([])

# --- colorbar (far right, in its own axis) ---
cbar = fig.colorbar(im, cax=ax_cb)
cbar.set_label("Valence (0 = neg → 1 = pos)")
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

plt.show()


In [ ]:
# --- fit & evaluate ---
reg = Ridge(alpha=1).fit(X_train, y_train_reg) # alpha=1.0 is the default regularization strength. 
y_pred_reg = reg.predict(X_test)

mae  = mean_absolute_error(y_test_reg, y_pred_reg)
r, _ = pearsonr(y_test_reg, y_pred_reg)
print(f"\nMAE       : {mae:.3f}    (baseline = predicting the train mean would give ~{np.mean(np.abs(y_test_reg - y_train_reg.mean())):.3f})")
print(f"Pearson r : {r:.3f}    (higher is better)")

In [ ]:
plt.figure(figsize=(7, 6))
sns.regplot(x=y_test_reg, y=y_pred_reg, scatter_kws={"alpha": 0.25, "s": 15})

x_lo, x_hi = y_test_reg.min(), y_test_reg.max()
y_lo, y_hi = y_pred_reg.min(), y_pred_reg.max()

mx = 0.05 * (x_hi - x_lo)
my = 0.05 * (y_hi - y_lo)
plt.xlim(x_lo - mx, x_hi + mx)
plt.ylim(y_lo - my, y_hi + my)

plt.xlabel("True Valence")
plt.ylabel("Predicted Valence")
plt.title(f"Regression — Pearson r = {r:.2f}")
plt.legend(); plt.tight_layout(); plt.show()


## Formulation 3 — Three-way: low / medium / high

Sometimes binary is too coarse and a continuous score is hard to interpret. A three-class formulation — *"is the sentiment low, medium, or high?"* — is often the right middle ground.

Decisions for `y`:

- **Where to put the bin edges?** With Valence on a 1-5 scale, a arbitrary fixed split is **`V ≤ 2.6` → low, `2.6 < V < 3.4` → medium, `V ≥ 3.4` → high**.
- It's classification with 3 classes → `y` is integers in `{0, 1, 2}`.

In real applications you can also choose bin edges *empirically* — e.g., the 33rd and 67th percentiles of the continuous score, so each bucket has roughly equal counts. We'll show that variation too.

In [ ]:
def valence_to_three(v, low=2.6, high=3.4):
    """V ≤ low → 0 (low),  low < V < high → 1 (mid),  V ≥ high → 2 (high)"""
    return np.where(v <= low, 0, np.where(v >= high, 2, 1))

y_train_3 = valence_to_three(v_train)
y_test_3  = valence_to_three(v_test)

print("Class balance (train):", dict(zip(*np.unique(y_train_3, return_counts=True))))
print("Class balance (test): ", dict(zip(*np.unique(y_test_3,  return_counts=True))))

# class_weight="balanced" — EmoBank is heavily skewed toward "mid", so without re-weighting the model would just predict "mid" for nearly everything.
clf3 = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train_3)
y_pred_3 = clf3.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test_3, y_pred_3):.3f}")
print(classification_report(y_test_3, y_pred_3, target_names=["low", "mid", "high"]))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_3, y_pred_3, normalize="true")
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=["low", "mid", "high"],
            yticklabels=["low", "mid", "high"])
plt.xlabel("Predicted");  plt.ylabel("True")
plt.title("3-way sentiment — confusion matrix")
plt.tight_layout(); plt.show()

### cf. percentile-based bin edges

If you don't want to hard-code cutoffs, you can let the data decide. Pick the 33rd and 67th percentile of the continuous score so each bin gets roughly a third of the data.

In [ ]:
# Use the regression target as the continuous score to bucket
edges = np.quantile(y_train_reg, [1/3, 2/3])
print(f"Bin edges (33rd, 67th pct of train): {edges.round(3)}")

def bucket(score, edges):
    return np.digitize(score, edges)   # → 0 / 1 / 2

y_train_3p = bucket(y_train_reg, edges)
y_test_3p  = bucket(y_test_reg,  edges)

print("Train balance:", dict(zip(*np.unique(y_train_3p, return_counts=True))))

clf3p = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train_3p)
print(f"\nAccuracy (percentile bins): {accuracy_score(y_test_3p, clf3p.predict(X_test)):.3f}")

In [ ]:
# Visualize the two binning strategies (fixed vs percentile) to see how well they separate the signal.
import matplotlib.pyplot as plt
import numpy as np

v_train_raw = y_train_reg * 4 + 1   # [0,1] → [1,5] 역변환

# bin edges
fixed_edges_raw = np.array([2.6, 3.4])
pct_edges_norm  = np.quantile(y_train_reg, [1/3, 2/3])
pct_edges_raw   = pct_edges_norm * 4 + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# --- (1) Fixed bins ---
axes[0].hist(v_train_raw, bins=50, color="#cccccc", edgecolor="white")
for e in fixed_edges_raw:
    axes[0].axvline(e, color="crimson", lw=2)
axes[0].axvspan(v_train_raw.min(), fixed_edges_raw[0], alpha=0.15, color="red",   label="low")
axes[0].axvspan(fixed_edges_raw[0], fixed_edges_raw[1], alpha=0.15, color="gray", label="mid")
axes[0].axvspan(fixed_edges_raw[1], v_train_raw.max(),  alpha=0.15, color="green", label="high")
axes[0].set_title(f"Fixed bins  (edges = {fixed_edges_raw.tolist()})\n"
                  f"bin gap from neutral = ±0.4")
axes[0].set_xlabel("Valence")
axes[0].set_ylabel("# sentences")
axes[0].legend(loc="upper right")

# --- (2) Percentile bins ---
axes[1].hist(v_train_raw, bins=50, color="#cccccc", edgecolor="white")
for e in pct_edges_raw:
    axes[1].axvline(e, color="crimson", lw=2)
axes[1].axvspan(v_train_raw.min(), pct_edges_raw[0], alpha=0.15, color="red",   label="low")
axes[1].axvspan(pct_edges_raw[0], pct_edges_raw[1], alpha=0.15, color="gray", label="mid")
axes[1].axvspan(pct_edges_raw[1], v_train_raw.max(),  alpha=0.15, color="green", label="high")
axes[1].set_title(f"Percentile bins  (edges = {pct_edges_raw.round(2).tolist()})\n"
                  f"bin gap from neutral = ±{(pct_edges_raw[1]-pct_edges_raw[0])/2:.2f}")
axes[1].set_xlabel("Valence")
axes[1].legend(loc="upper right")

plt.suptitle("Same data, two ways to draw class boundaries", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


# Part 3 — Modern: Fine-tuning a pre-trained Transformer

The models in Part 2 used a hand-built feature pipeline (`text → tf-idf → classifier model → evaluate`). State-of-the-art sentiment models skip that pipeline and instead **fine-tune a large pre-trained Transformer** (BERT, RoBERTa, DistilBERT, …) end-to-end on the labeled data.

## The big picture

To fine-tune a model, we need to assemble four training ingredients like below and hand them to a training loop:

```
   ┌─────────────────────┐
   │  1. MODEL           │   pre-trained model (e.g., transformer) + a fresh classification head
   ├─────────────────────┤
   │  2. DATA            │   tokenized text (X) + labels (y)
   ├─────────────────────┤   
   │  3. HYPERPARAMETERS │   number of epochs, learning rate, batch size, …
   ├─────────────────────┤
   │  4. METRIC          │   how to score the model on the eval set
   └─────────────────────┘
```

In principle each of these is a sizable engineering task — designing the model architecture, writing the tokenizer, implementing backprop and the optimizer, building the training/eval loop. **HuggingFace's `transformers` library reduces each one to a single function call**, and the `Trainer` class glues them together so you don't have to write the loop yourself.

The next few cells walk through these four ingredients in order. You don't need to understand the internals — just recognize the *role* each piece plays, and you'll be able to read and adapt most fine-tuning code you find online.

## A few terms to recognize as you read

- **Transformer** — the neural-net architecture behind BERT and GPT. 
- **Pre-training vs. fine-tuning** — pre-training is the expensive step that someone else (Google, Meta, …) already did on huge text corpora; fine-tuning is the cheap step we do on top, on our small labeled dataset. 
- **Classification head** — a small new layer added on top of the Transformer that maps its output to our 2 classes. Initialized randomly, trained together with the rest.
- **Epoch / batch / learning rate** — standard deep-learning training knobs. We'll set them via `TrainingArguments`.
- **GPU** — training a 66M-parameter model on CPU is painfully slow. Colab gives you a free GPU under *Runtime → Change runtime type*.

In [ ]:
# !pip install transformers datasets accelerate torch
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch sees GPU: {torch.cuda.is_available()}")

## Step 1 — Reshape the data into a HuggingFace `Dataset`

The `transformers` library wants its data wrapped in a `datasets.Dataset` object (it's an Arrow-backed table with a few convenient methods like `.map` and `.set_format`). We reuse the **binary task** we built in Part 2: `V > 3` → positive, `V < 3` → negative, dropping the near-neutral band (`|V − 3| < 0.2`).

In [ ]:
# Recycle the binary labels and the train/test indices from Part 2.
train_rows = df.loc[train_idx][mask_train].copy()
test_rows  = df.loc[test_idx ][mask_test ].copy()

train_rows["label"] = (train_rows["valence"] > 3).astype(int)
test_rows ["label"] = (test_rows ["valence"] > 3).astype(int)

train_ds = Dataset.from_pandas(train_rows[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_rows [["text", "label"]], preserve_index=False)

print(train_ds)
print(test_ds)
print("\nOne example:", train_ds[100])

## Step 2 — Tokenize

The tokenizer turns each review into `input_ids` (integer subword tokens) and `attention_mask`. `attention_mask=1` marks real tokens; `0` marks the padding that fills up to `max_length=256

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

print("=== Before tokenization ===")
print("columns:", train_ds.column_names)
print("sample :", train_ds[100])

train_ds_tokenized = train_ds.map(tokenize, batched=True)
test_ds_tokenized  = test_ds.map(tokenize,  batched=True)

sample = train_ds_tokenized[100]
print("\n=== After tokenization ===")
print("columns:", train_ds_tokenized.column_names)
for key, value in sample.items():
    if isinstance(value, list) and len(value) > 20:
        print(f"  {key:16s}: {value[:20]} ...  (len={len(value)})")
    else:
        print(f"  {key:16s}: {value}")

# Trainer expects PyTorch tensors
train_ds_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds_tokenized.set_format ("torch", columns=["input_ids", "attention_mask", "label"])

sample = train_ds_tokenized[100]
print("\n=== Diagnostics ===")
print("input_ids  (first 20):", sample["input_ids"][:20].tolist())
print("decoded back         :", tokenizer.decode(sample["input_ids"][:20]))
print(f"attention_mask sum   : {sample['attention_mask'].sum().item()} out of {len(sample['attention_mask'])}")


## Step 3 — Load the pre-trained model + add a classification head

`AutoModelForSequenceClassification` loads the pre-trained DistilBERT weights and **attaches a fresh linear classification head** with `num_labels` outputs. The head is randomly initialized — that's the part that *must* be trained.

This is exactly what "fine-tuning" means here: the bottom (the pre-trained Transformer) already knows language; the top (the new head) is blank. During training, gradients flow through both — the head learns from scratch, the Transformer's weights get small, targeted updates that adapt its general language knowledge to your specific task.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 4 — `TrainingArguments` (the hyperparameters)

This object collects everything about *how* training happens. The four knobs that matter most:

| Argument | What it controls | Typical for fine-tuning |
|---|---|---|
| `num_train_epochs` | how many passes through the training set | 2–4 |
| `learning_rate` | how big each optimizer step is | `2e-5` to `5e-5` |
| `per_device_train_batch_size` | how many examples per gradient update | 8–32 (limited by GPU memory) |
| `eval_strategy` | when to evaluate ("epoch" = end of each epoch) | `"epoch"` |


In [ ]:
args = TrainingArguments(
    output_dir="../bert_emobank_out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no",        # don't checkpoint to disk — keep things lean
    report_to="none",
)

## Step 5 — `Trainer`: glue everything together, then train

`Trainer` wraps the model, the data, the args, and the metric function. When you call `.train()` it runs the full loop:

```
for epoch in range(num_train_epochs):
    for batch in train_dataloader:        # shuffled batches
        outputs = model(**batch)          # forward pass → loss
        loss.backward()                   # backward pass → gradients
        optimizer.step()                  # AdamW updates parameters
        optimizer.zero_grad()
    evaluate on test set
```

You don't write any of that by hand — the `Trainer` does it. But that's what's happening under the hood.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds_tokenized,
    eval_dataset=test_ds_tokenized,
    compute_metrics=compute_metrics,
)

trainer.train()

## Step 6 — Evaluate and compare with the tf-idf baseline

In [ ]:
bert_eval = trainer.evaluate()
print(bert_eval)

# Re-evaluate the Part-2 logistic regression on the same (near-neutral-dropped) test set
from sklearn.metrics import accuracy_score, f1_score
y_pred_lr = clf_bin.predict(X_test_bin)

print("\n--- Comparison on the binary EmoBank test set ---")
print(f"Tf-idf + Logistic Regression : acc = {accuracy_score(y_test_bin, y_pred_lr):.3f}   f1 = {f1_score(y_test_bin, y_pred_lr):.3f}")
print(f"Fine-tuned DistilBERT        : acc = {bert_eval['eval_accuracy']:.3f}   f1 = {bert_eval['eval_f1']:.3f}")

## Step 7 — Inference on new text

Once trained, the model is just like any HuggingFace pipeline — give it text, get back predictions.

In [ ]:
from transformers import pipeline

clf_pipe = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer, device=0 if device == "cuda" else -1)

new_texts = [
    "What a beautiful sunset over the harbor tonight.",
    "I am absolutely devastated by the news.",
    "The package arrived. It was as described.",
    "She smiled and thanked everyone for coming.",
    "He shouted at the children until they cried.",
]
for t in new_texts:
    out = clf_pipe(t)[0]
    print(f"  {out['label']:8s}  ({out['score']:.2f})  |  {t}")

# Final takeaways — when does fine-tuning help?

### When it's worth it
- You have at least a few thousand labeled examples.
- The task is *linguistic* — sentiment, stance, topic, entailment — where word order and context matter (and where bag-of-words throws away too much information).
- You can run on a GPU (Colab free tier is enough for models up to ~DistilBERT size).

### When you should *not* reach for BERT
- You only have a few hundred labels — a good off-the-shelf model + careful prompt may beat a fine-tuned BERT.
- The signal is mostly in metadata, not in the text — tf-idf or even raw features will do fine and be 1000× faster.
- You need interpretability — linear models on tf-idf tell you exactly which words drive each prediction; a fine-tuned Transformer does not (you need extra tools like attention visualization or SHAP).